In [1]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


/home/mert/.local/lib/python3.10/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


In [3]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
C   0.000   0.691   1.172
C   0.000  -0.691   1.172
C   0.000  -1.321  -0.065
C   0.000   1.321  -0.065
H   0.000   1.253   2.090
H   0.000  -1.253   2.090
H   0.000  -2.395  -0.141
H   0.000   2.395  -0.141
N   0.000   0.654  -1.227
N   0.000  -0.654  -1.227'''
molecule = 'o_C4H4N2'
spin = 0
charge =  0


In [4]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 5 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [5]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [6]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

output file: ./o_C4H4N2.txt
orbital  energy   occ
0     -15.591966130469281    2.0
1     -15.591823848859248    2.0
2     -11.245533719688    2.0
3     -11.245448292922799    2.0
4     -11.230340207574303    2.0
5     -11.229646678070251    2.0
6     -1.3109737771685652    2.0
7     -1.067088602276817    2.0
8     -1.0621505475405566    2.0
9     -0.8499284575429757    2.0
10     -0.834094791906434    2.0
11     -0.6924134846604212    2.0
12     -0.6340534652307775    2.0
13     -0.627395313240283    2.0
14     -0.5579782174860204    2.0
15     -0.5358694854577813    2.0
16     -0.5338084053869618    2.0
17     -0.4133951329603055    2.0
18     -0.35477259720302146    2.0
19     -0.3384344815590076    2.0
20     -0.3186866789304694    2.0
21     0.20769627220670023    0.0
22     0.2192104419615813    0.0
23     0.449741168356226    0.0
24     0.5607176832839339    0.0
25     0.606553775520962    0.0
26     0.6673509871988477    0.0
27     0.6865344588304875    0.0
28     0.697385762041

In [7]:
n_elec= 6
n_orb = 6 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([17,19,20,22,23,24])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-262.0163089810383),
 np.float64(-7.0376749472377185),
 FCIvector([[-9.11358301e-01,  5.79696124e-03,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00, -9.20169060e-03,
             -5.25794969e-03, -8.29179091e-02, -1.17063259e-02,
              0.00000000e+00,  1.57848590e-03,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
              2.25052835e-02,  2.78945206e-03, -2.19076415e-02,
              0.00000000e+00,  0.00000000e+00],
            [ 5.79696124e-03,  3.03116421e-02,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00, -1.17925532e-01,
             -2.51284246e-02,  4.88809331e-03, -8.39567642e-03,
              0.00000000e+00, -8.60325584e-03,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
             -5.14175743e-04,  9.86213580e-03,  2.82403596e-03,
              0.00000000e+00,  0.00000000e+00],
            [ 0.00000000e+00,  0.00000000e+00,  1.76197487e-01,
     

In [8]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0 1 2]      [0 1 2]      -0.911358301469224
[0 1 2]      [0 1 3]      0.005796961237546
[0 1 2]      [0 2 4]      -0.009201690597428
[0 1 2]      [1 2 4]      -0.005257949686722
[0 1 2]      [0 3 4]      -0.082917909067174
[0 1 2]      [1 3 4]      -0.011706325947000
[0 1 2]      [0 1 5]      0.001578485899928
[0 1 2]      [2 3 5]      0.022505283490859
[0 1 2]      [0 4 5]      0.002789452060624
[0 1 2]      [1 4 5]      -0.021907641480316
[0 1 3]      [0 1 2]      0.005796961237546
[0 1 3]      [0 1 3]      0.030311642124308
[0 1 3]      [0 2 4]      -0.117925532341804
[0 1 3]      [1 2 4]      -0.025128424607435
[0 1 3]      [0 3 4]      0.004888093308958
[0 1 3]      [1 3 4]      -0.008395676419048
[0 1 3]      [0 1 5]      -0.008603255839474
[0 1 3]      [2 3 5]      -0.000514175742527
[0 1 3]      [0 4 5]      0.009862135803613
[0 1 3]      [1 4 5]      0.002824035961150
[0 2 3]      [0 2 3]      0.176197487478175
[0 2 3]      [1 2 3]      0.023999620918970
[0 2 3]      [0 1 4] 